In [1]:
import os
from typing import TypedDict, Annotated, Literal
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage
from langchain_google_genai import ChatGoogleGenerativeAI

C:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
API_KEY = 'API_KEY'

In [3]:
llm = ChatGoogleGenerativeAI(
    model = 'gemini-2.5-flash',
    google_api_key = API_KEY,
    temperature = 0.2
)

In [4]:
class TelecomState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    complaint: str

    # Stage 1 outputs
    issue_type: str
    severity: str
    scope: str
    sentiment: str
 
    # Stage 2 outputs
    tower_status: str
    crm_status: str
    billing_status: str
    device_status: str
    outage_status: str
 
    # Stage 3 output
    assigned_agent: str
 
    # Stage 4 outputs
    customer_response: str
    technician_notes: str
    estimated_resolution_time: str
    final_summary: str

In [5]:
def complaint_understanding_node(state: TelecomState) -> dict:
    complaint = state["complaint"]
    prompt = f"""You are a telecom complaint analysis system.
Complaint: "{complaint}"

Respond in EXACTLY this format (no extra text):
issue_type: <one of: outage | slow_internet | billing | sim | call_drop | congestion | device>
scope: <one of: individual | multiple_users | regional>
severity: <one of: low | medium | high | critical>
sentiment: <one of: calm | frustrated | angry>"""

    response = llm.invoke([HumanMessage(content=prompt)]).content.strip()
    parsed = {k.strip(): v.strip() for k, v in (line.split(":", 1) for line in response.splitlines() if ":" in line)}

    return {
        "issue_type": parsed.get("issue_type", "unknown"),
        "scope":      parsed.get("scope",      "individual"),
        "severity":   parsed.get("severity",   "medium"),
        "sentiment":  parsed.get("sentiment",  "calm"),
        "messages":   [AIMessage(content=f"[Stage 1] {parsed}")],
    }

In [6]:
def tower_health_check_node(state: TelecomState) -> dict:
    """Simulate querying tower health system."""
    issue_type = state.get("issue_type", "unknown")
    scope = state.get("scope", "individual")
 
    if issue_type in ("outage", "congestion", "call_drop") or scope == "regional":
        status = "Tower B-42 reporting degraded signal. Maintenance ticket #TW-9921 raised."
    else:
        status = "All towers operating normally in requested area."
 
    return {
        "tower_status": status,
        "messages": [AIMessage(content=f"[Tower Health] {status}")],
    }
 
 
def crm_check_node(state: TelecomState) -> dict:
    """Simulate querying CRM/ticketing system."""
    complaint = state["complaint"]
    # Simulated: check if customer has repeated tickets
    if "again" in complaint.lower() or "third time" in complaint.lower():
        status = "Customer has 3 prior open tickets. Flagged as repeat complaint — priority escalation required."
    else:
        status = "No prior open tickets found for this customer."
 
    return {
        "crm_status": status,
        "messages": [AIMessage(content=f"[CRM Check] {status}")],
    }
 
 
def billing_system_check_node(state: TelecomState) -> dict:
    """Simulate querying billing system."""
    issue_type = state.get("issue_type", "unknown")
 
    if issue_type == "billing":
        status = "Billing anomaly detected: Customer charged ₹499 on 2024-12-01 with no active data session logged."
    else:
        status = "No billing anomalies found."
 
    return {
        "billing_status": status,
        "messages": [AIMessage(content=f"[Billing System] {status}")],
    }
 
 
def device_compatibility_check_node(state: TelecomState) -> dict:
    """Simulate querying device compatibility database."""
    issue_type = state.get("issue_type", "unknown")
 
    if issue_type in ("sim", "device"):
        status = "SIM card reports ICCID mismatch. Possible SIM deactivation post-recharge due to system sync delay."
    else:
        status = "Device/SIM compatibility — no issues detected."
 
    return {
        "device_status": status,
        "messages": [AIMessage(content=f"[Device Check] {status}")],
    }
 
 
def outage_monitoring_node(state: TelecomState) -> dict:
    """Simulate querying outage monitoring API."""
    scope = state.get("scope", "individual")
    issue_type = state.get("issue_type", "unknown")
 
    if scope == "regional" or issue_type == "outage":
        status = "ACTIVE OUTAGE: Zone 4-B (North District). Approx. 1,200 subscribers affected. ETA: 3 hours."
    else:
        status = "No active outage alerts for the customer's registered area."
 
    return {
        "outage_status": status,
        "messages": [AIMessage(content=f"[Outage Monitor] {status}")],
    }

In [7]:
def routing_decision_node(state: TelecomState) -> dict:
    """Determine which specialist agent should handle the case."""
    issue_type = state.get("issue_type", "unknown")
    severity = state.get("severity", "medium")
    crm_status = state.get("crm_status", "")
    scope = state.get("scope", "individual")
 
    if severity == "critical" or scope == "regional":
        agent = "escalation_agent"
    elif "repeat complaint" in crm_status.lower() or "prior open tickets" in crm_status.lower() and "3" in crm_status:
        agent = "priority_handling_agent"
    elif issue_type == "billing":
        agent = "billing_agent"
    elif issue_type in ("outage", "congestion", "call_drop"):
        agent = "network_ops_agent"
    elif issue_type in ("sim", "device"):
        agent = "device_support_agent"
    else:
        agent = "general_support_agent"
 
    return {
        "assigned_agent": agent,
        "messages": [AIMessage(content=f"[Router] Routing to → {agent}")],
    }
 
 
def router_edge(state: TelecomState) -> Literal[
    "billing_agent", "network_ops_agent", "device_support_agent",
    "escalation_agent", "priority_handling_agent", "general_support_agent"
]:
    return state["assigned_agent"]

In [8]:
def _run_specialist(role: str, context: dict) -> str:
    prompt = f"""You are the {role} for a telecom company.
Context:
{chr(10).join(f"- {k}: {v}" for k, v in context.items())}
 
Provide a brief internal action note (2-3 sentences) on what this agent does next."""
    return llm.invoke([HumanMessage(content=prompt)]).content.strip()
 
 
def billing_agent_node(state: TelecomState) -> dict:
    note = _run_specialist("Billing Resolution Agent", {
        "complaint": state["complaint"],
        "billing_status": state.get("billing_status"),
        "severity": state.get("severity"),
    })
    return {"assigned_agent": "billing_agent", "messages": [AIMessage(content=f"[Billing Agent] {note}")]}
 
 
def network_ops_agent_node(state: TelecomState) -> dict:
    note = _run_specialist("Network Operations Agent", {
        "complaint": state["complaint"],
        "tower_status": state.get("tower_status"),
        "outage_status": state.get("outage_status"),
        "severity": state.get("severity"),
    })
    return {"assigned_agent": "network_ops_agent", "messages": [AIMessage(content=f"[Network Ops Agent] {note}")]}
 
 
def device_support_agent_node(state: TelecomState) -> dict:
    note = _run_specialist("Device & SIM Support Agent", {
        "complaint": state["complaint"],
        "device_status": state.get("device_status"),
    })
    return {"assigned_agent": "device_support_agent", "messages": [AIMessage(content=f"[Device Support Agent] {note}")]}
 
 
def escalation_agent_node(state: TelecomState) -> dict:
    note = _run_specialist("High-Severity Escalation Agent", {
        "complaint": state["complaint"],
        "scope": state.get("scope"),
        "outage_status": state.get("outage_status"),
        "severity": state.get("severity"),
    })
    return {"assigned_agent": "escalation_agent", "messages": [AIMessage(content=f"[Escalation Agent] {note}")]}
 
 
def priority_handling_agent_node(state: TelecomState) -> dict:
    note = _run_specialist("Priority Handling Agent (Repeat Complaints)", {
        "complaint": state["complaint"],
        "crm_status": state.get("crm_status"),
    })
    return {"assigned_agent": "priority_handling_agent", "messages": [AIMessage(content=f"[Priority Agent] {note}")]}
 
 
def general_support_agent_node(state: TelecomState) -> dict:
    note = _run_specialist("General Customer Support Agent", {
        "complaint": state["complaint"],
        "issue_type": state.get("issue_type"),
    })
    return {"assigned_agent": "general_support_agent", "messages": [AIMessage(content=f"[General Support Agent] {note}")]}

In [9]:
def customer_response_node(state: TelecomState) -> dict:
    """Generate customer-facing response."""
    prompt = f"""You are a telecom customer support AI.
Complaint: "{state['complaint']}"
Issue Type: {state.get('issue_type')}
Severity: {state.get('severity')}
Tower Status: {state.get('tower_status')}
Outage Status: {state.get('outage_status')}
Billing Status: {state.get('billing_status')}
Device Status: {state.get('device_status')}
Assigned Agent: {state.get('assigned_agent')}
 
Write a polite, empathetic customer-facing message (3-4 sentences) acknowledging the issue and what action is being taken."""
 
    response = llm.invoke([HumanMessage(content=prompt)]).content.strip()
    return {
        "customer_response": response,
        "messages": [AIMessage(content=f"[Customer Response]\n{response}")],
    }
 
 
def technician_notes_node(state: TelecomState) -> dict:
    """Generate internal technician notes."""
    prompt = f"""Generate concise internal technician notes for this telecom incident.
Complaint: "{state['complaint']}"
Issue: {state.get('issue_type')} | Severity: {state.get('severity')} | Scope: {state.get('scope')}
Tower: {state.get('tower_status')}
Outage: {state.get('outage_status')}
Billing: {state.get('billing_status')}
Device: {state.get('device_status')}
CRM: {state.get('crm_status')}
Agent: {state.get('assigned_agent')}
 
Provide bullet-point technician notes."""
 
    response = llm.invoke([HumanMessage(content=prompt)]).content.strip()
    return {
        "technician_notes": response,
        "messages": [AIMessage(content=f"[Technician Notes]\n{response}")],
    }
 
 
def eta_prediction_node(state: TelecomState) -> dict:
    """Predict estimated resolution time."""
    severity = state.get("severity", "medium")
    scope = state.get("scope", "individual")
    issue_type = state.get("issue_type", "unknown")
 
    eta_map = {
        ("critical", "regional"): "4-6 hours (field crew dispatched)",
        ("high", "multiple_users"): "2-4 hours",
        ("high", "individual"): "1-2 hours",
        ("medium", "individual"): "30-60 minutes",
        ("low", "individual"): "15-30 minutes",
    }
    eta = eta_map.get((severity, scope), "1-2 hours")
    if issue_type == "billing":
        eta = "24-48 hours (billing cycle verification required)"
 
    return {
        "estimated_resolution_time": eta,
        "messages": [AIMessage(content=f"[ETA Prediction] Estimated resolution: {eta}")],
    }

In [10]:
def final_summary_node(state: TelecomState) -> dict:
    """Compile final summary."""
    summary = f"""
========== TELECOM INCIDENT RESOLUTION SUMMARY ==========
Complaint      : {state['complaint']}
Issue Type     : {state.get('issue_type')}
Severity       : {state.get('severity')}
Scope          : {state.get('scope')}
Sentiment      : {state.get('sentiment')}
Assigned Agent : {state.get('assigned_agent')}
ETA            : {state.get('estimated_resolution_time')}
 
--- Backend Checks ---
Tower   : {state.get('tower_status')}
Outage  : {state.get('outage_status')}
Billing : {state.get('billing_status')}
Device  : {state.get('device_status')}
CRM     : {state.get('crm_status')}
 
--- Customer Response ---
{state.get('customer_response')}
 
--- Technician Notes ---
{state.get('technician_notes')}
========================================================="""
 
    return {
        "final_summary": summary,
        "messages": [AIMessage(content=summary)],
    }
 

In [11]:
def build_telecom_graph() -> StateGraph:
    graph = StateGraph(TelecomState)

    # Stage 1
    graph.add_node("complaint_understanding", complaint_understanding_node)

    # Stage 2
    graph.add_node("tower_health_check", tower_health_check_node)
    graph.add_node("crm_check", crm_check_node)
    graph.add_node("billing_system_check", billing_system_check_node)
    graph.add_node("device_compatibility_check", device_compatibility_check_node)
    graph.add_node("outage_monitoring", outage_monitoring_node)

    # Stage 3
    graph.add_node("routing_decision", routing_decision_node)
    graph.add_node("billing_agent", billing_agent_node)
    graph.add_node("network_ops_agent", network_ops_agent_node)
    graph.add_node("device_support_agent", device_support_agent_node)
    graph.add_node("escalation_agent", escalation_agent_node)
    graph.add_node("priority_handling_agent", priority_handling_agent_node)
    graph.add_node("general_support_agent", general_support_agent_node)

    # Stage 4
    graph.add_node("customer_response", customer_response_node)
    graph.add_node("technician_notes", technician_notes_node)
    graph.add_node("eta_prediction", eta_prediction_node)
    graph.add_node("final_summary", final_summary_node)

    # START → Stage 1
    graph.add_edge(START, "complaint_understanding")

    # Stage 1 → Stage 2 (fan-out)
    for s2 in ["tower_health_check", "crm_check", "billing_system_check",
               "device_compatibility_check", "outage_monitoring"]:
        graph.add_edge("complaint_understanding", s2)

    # Stage 2 → Routing (fan-in)
    for s2 in ["tower_health_check", "crm_check", "billing_system_check",
               "device_compatibility_check", "outage_monitoring"]:
        graph.add_edge(s2, "routing_decision")

    # Stage 3 — conditional routing
    graph.add_conditional_edges(
        "routing_decision",
        router_edge,
        {
            "billing_agent": "billing_agent",
            "network_ops_agent": "network_ops_agent",
            "device_support_agent": "device_support_agent",
            "escalation_agent": "escalation_agent",
            "priority_handling_agent": "priority_handling_agent",
            "general_support_agent": "general_support_agent",
        },
    )

    # Specialist agents → Stage 4 (fan-out)
    specialist_agents = [
        "billing_agent", "network_ops_agent", "device_support_agent",
        "escalation_agent", "priority_handling_agent", "general_support_agent",
    ]
    for agent in specialist_agents:
        for s4 in ["customer_response", "technician_notes", "eta_prediction"]:
            graph.add_edge(agent, s4)

    # Stage 4 → Final Summary (fan-in)
    for s4 in ["customer_response", "technician_notes", "eta_prediction"]:
        graph.add_edge(s4, "final_summary")

    graph.add_edge("final_summary", END)

    return graph.compile()

In [12]:
def run_telecom_system(complaint: str):
    print(f"COMPLAINT: {complaint}")
 
    app = build_telecom_graph()
    initial_state: TelecomState = {
        "messages": [HumanMessage(content=complaint)],
        "complaint": complaint,
        "issue_type": "",
        "severity": "",
        "scope": "",
        "sentiment": "",
        "tower_status": "",
        "crm_status": "",
        "billing_status": "",
        "device_status": "",
        "outage_status": "",
        "assigned_agent": "",
        "customer_response": "",
        "technician_notes": "",
        "estimated_resolution_time": "",
        "final_summary": "",
    }
 
    result = app.invoke(initial_state)
    print(result["final_summary"])

In [13]:
if __name__ == "__main__":
    complaints = [
        "Internet is extremely slow since morning."
    ]
    for complaint in complaints:
        run_telecom_system(complaint)
        print("\n")

COMPLAINT: Internet is extremely slow since morning.

========== TELECOM INCIDENT RESOLUTION SUMMARY ==========
Complaint      : Internet is extremely slow since morning.
Issue Type     : slow_internet
Severity       : high
Scope          : individual
Sentiment      : frustrated
Assigned Agent : general_support_agent
ETA            : 1-2 hours

--- Backend Checks ---
Tower   : All towers operating normally in requested area.
Outage  : No active outage alerts for the customer's registered area.
Billing : No billing anomalies found.
Device  : Device/SIM compatibility — no issues detected.
CRM     : No prior open tickets found for this customer.

--- Customer Response ---
I'm very sorry to hear your internet has been extremely slow since morning; I understand how frustrating this can be. Please rest assured, we're actively looking into the root cause of the slow speeds you're experiencing. We're initiating a diagnostic check on your connection right away to identify and resolve the issue.